1. Setup weave library
```python
pip install wandb weave
```
2. Visit https://wandb.ai and sign up
3. Ensure you are in free tier, if not then downgrade your plan
4. Create a new project which be use later tutorial: llm-eval-dummy.
5. Go to https://wandb.ai/authorize and create new API Key
6. In command line, login with your api key. Enter command `wandb login`
7. Use `wandb login --relogin` to force relogin(optional)

In [1]:
import os, asyncio, re
import weave
from weave import Model, Dataset, Evaluation
from openai import OpenAI
from dotenv import load_dotenv
from weave.flow import leaderboard
from weave.trace.ref_util import get_ref

c:\Users\wengshang.hoo\AppData\Local\miniconda3\envs\ai_sl\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
load_dotenv()

weave.init('jiadexx-sophic/llm-eval-dummy')

weave: Logged in as Weights & Biases user: jiadexx.
weave: View Weave data at https://wandb.ai/jiadexx-sophic/llm-eval-dummy/weave


In [ ]:
# ---- 2) Tiny evaluation dataset ---------------------------------------------
# Each row contains: a question, a short context that actually contains the answer,
# and a reference answer we consider "correct".
rows = [
    {
        "id": "1",
        "question": "What is the capital of Spain?",
        "context": "Spain's capital is Madrid.",
        "reference": "Madrid"
    },
    {
        "id": "2",
        "question": "Who wrote Pride and Prejudice?",
        "context": "The novel Pride and Prejudice was authored by Jane Austen in 1813.",
        "reference": "Jane Austen"
    },
    {
        "id": "3",
        "question": "What year did Apollo 11 land on the Moon?",
        "context": "Apollo 11 landed on the Moon in 1969, with Armstrong and Aldrin walking on the surface.",
        "reference": "1969"
    },
    {
        "id": "4",
        "question": "Which element has the chemical symbol 'Na'?",
        "context": "In the periodic table, 'Na' denotes sodium, which forms NaCl with chlorine.",
        "reference": "Sodium"
    },
    {
        "id": "5",
        "question": "Name the largest planet in our solar system.",
        "context": "Jupiter is the largest planet in our solar system.",
        "reference": "Jupiter"
    }
]

# Use a versioned Weave Dataset object (nice for UI & versioning)
dataset = Dataset(name="qa_eval_v1", rows=rows)
weave.publish(dataset)  # makes it appear/version in the Weave UI

weave: 📦 Published to https://wandb.ai/jiadexx-sophic/llm-eval-dummy/weave/objects/qa_eval_v1/versions/7s41Drsb7iWrNIcY08W9JwR3ZD8fiPpId9DlVS3XYo4


ObjectRef(entity='jiadexx-sophic', project='llm-eval-dummy', name='qa_eval_v1', _digest='7s41Drsb7iWrNIcY08W9JwR3ZD8fiPpId9DlVS3XYo4', _extra=())

After publish, checkout the dataset at: https://wandb.ai/your_username/your_project/weave/datasets

Or you can find it at **Assets** menu

In [4]:
# ---- 3) Two model variants to compare ---------------------------------------
client = OpenAI(
    base_url="http://localhost:11434/v1", # OpenAI 兼容接口（你现在用的）
    api_key="ollama", # 随便填，Ollama 不需要真实 Key
)

# Variant A: permissive prompt (more likely to hallucinate if context is weak)
# permission prompt: Less restriction and more freedom to llm.
class QAModelLoose(Model):
    model_name: str = "qwen3:1.7b"
    system_prompt: str = (
        "You are a helpful assistant. Answer the user question clearly."
    )
    @weave.op()
    def predict(self,question:str,context:str) -> str:
        msg = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": f"Question: {question}\nContext:\n{context}"},
        ]
        res = client.chat.completions.create(
            model=self.model_name, 
            messages=msg, 
            temperature=0.2)
        return res.choices[0].message.content.strip()
    
# Variant B: grounded prompt (only answer from the given context)
class QAModelGrounded(Model):
    model_name: str = "qwen3:1.7b"
    system_prompt: str = (
        "You are a precise QA bot. Use ONLY the provided context. "
        "Give the shortest possible answer. ONE WORD or ONE PHRASE only. "
        "Do NOT use full sentences. Do NOT repeat the question."
    )
    @weave.op()
    def predict(self,question:str,context:str) -> str:
        msg = [
            {"role": "system", "content":self.system_prompt},
            {"role": "user", "content": f"""
                Answer the question using ONLY the context below.
                
                Context:
                {context}

                Question:
                {question}
                """
            }
        ]
        res = client.chat.completions.create(
            model=self.model_name,
            messages=msg,
            temperature=0.0
        )
        return res.choices[0].message.content.strip()

**Parameter Sources: Three-Way Meeting**

When running each row of test data, Weave automatically searches for data from the following three sources to populate your `function_name` parameter:

1. Dataset Columns: If your Dataset has `question`, `context`, and `reference` columns, the Scorer can directly request these parameters.
2. Model Output: By default, the result returned by the model's predict function is named `output`.
3. Automatic Mapping: Weave's internal logic automatically matches these names.

In [ ]:
# ---- 4) Define evaluation metrics (Scorers) ---------------------------------
# 4a) Heuristic: Exact match (case/space-insensitive)
@weave.op()
def exact_match(reference: str, output: str) -> dict:
    def norm(s: str) -> str:
        return re.sub(r"\s+", " ", s).strip().lower()
    return {"exact_match": norm(reference) == norm(output)}

# 4b) Heuristic: Jaccard token overlap
@weave.op()
def jaccard(reference: str, output: str) -> dict:
    a, b = set(reference.lower().split()), set(output.lower().split())
    return {"jaccard": (len(a & b) / len(a | b)) if (a | b) else 0.0}

# 4c) Built‑in: Embedding similarity (semantic match to reference)
import json

from weave.scorers import EmbeddingSimilarityScorer
sim_scorer = EmbeddingSimilarityScorer(
    model_id="ollama/nomic-embed-text",  # any LiteLLM-supported embedding model
    threshold=0.7,
    column_map={"target": "reference"}  # our dataset uses 'reference' instead of 'target'
)

# 4d) Built‑in: Hallucination‑free check (judge if answer invents facts not in context)
from weave.scorers import HallucinationFreeScorer
class StableHallucinationScorer(HallucinationFreeScorer):

    @weave.op()
    async def score(self, *, context: str, output: str, **kwargs):
        prompt = f"""
        You are a strict evaluator.

        Context:
        {context}

        Answer:
        {output}

        Return ONLY valid JSON:
        {{"has_hallucination": true or false}}
        """

        for _ in range(2):  # retry（防止空输出）
            res = client.chat.completions.create(
                model=self.model_id,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=128,
            )

            text = res.choices[0].message.content.strip()
            if text:
                break
        
        print(f"Hallucination output: {text}")

        try:
            return json.loads(text)
        except:
            return {
                "has_hallucination": None,
                "error": f"Invalid JSON: {text}"
            }

custom_hallucination_scorer = StableHallucinationScorer(
    model_id="qwen3.5:2b",
    # This scorer expects 'context' and 'output'; we map if names differ.
    column_map={"context": "context"},  # (output is auto-mapped)
    temperature=0,
    max_tokens=256
)

hallucination_scorer = HallucinationFreeScorer(
    model_id="openai/gpt-4o",
    # This scorer expects 'context' and 'output'; we map if names differ.
    column_map={"context": "context"}  # (output is auto-mapped)
)

import sacrebleu
from rouge_score import rouge_scorer

@weave.op()
def bleu(reference: str, output: str) -> dict:
    # sacrebleu expects: hypothesis string, list of reference strings
    score = sacrebleu.sentence_bleu(
        output,
        [reference],
        smooth_method="exp",
        tokenize="13a",   # default tokenizer; you can omit this line if you want
    ).score
    # sacrebleu returns 0–100; normalise to 0–1 for convenience
    return {"bleu": score / 100.0}

# 4d) ROUGE-L F1 (using rouge-score)
_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

@weave.op()
def rouge_l(reference: str, output: str) -> dict:
    scores = _rouge.score(reference, output)
    # F-measure is the usual scalar people report
    return {"rougeL_f": scores["rougeL"].fmeasure}

In [6]:
# ---- 5) Build the Evaluation -------------------------------------------------
evaluation = Evaluation(
    dataset=dataset,    # could also be `rows`
    scorers=[exact_match, jaccard,sim_scorer,hallucination_scorer]
)

In [7]:
# ---- 6) Run evaluations for both variants -----------------------------------
async def run_all():
    # You can optionally give runs a friendly display name for the UI
    await evaluation.evaluate(QAModelLoose(), __weave={"display_name": "loose_prompt"})
    await evaluation.evaluate(QAModelGrounded(), __weave={"display_name": "grounded_prompt"})

    spec = leaderboard.Leaderboard(
        name="qa_eval_leaderboard",
        description="Compare 'loose' vs 'grounded' prompts on QA metrics",
        columns=[
            leaderboard.LeaderboardColumn(
                evaluation_object_ref=get_ref(evaluation).uri(),
                scorer_name="EmbeddingSimilarityScorer",  # name of the scorer
                summary_metric_path="similarity_score.mean",  # choose any summary field
            ),
            leaderboard.LeaderboardColumn(
                evaluation_object_ref=get_ref(evaluation).uri(),
                scorer_name="HallucinationFreeScorer",
                summary_metric_path="has_hallucination.true_fraction",  # lower is better
            ),
            leaderboard.LeaderboardColumn(
                evaluation_object_ref=get_ref(evaluation).uri(),
                scorer_name="jaccard",
                summary_metric_path="jaccard.mean",
            ),
            leaderboard.LeaderboardColumn(
                evaluation_object_ref=get_ref(evaluation).uri(),
                scorer_name="exact_match",
                summary_metric_path="exact_match.true_fraction",
            ),
        ],
    )
    weave.publish(spec)

In [8]:
await run_all()

weave: 🍩 https://wandb.ai/jiadexx-sophic/llm-eval-dummy/r/call/019cff6f-0fc3-7b80-8424-7f20e1ea84d7
weave: Evaluated 1 of 5 examples
weave: Evaluated 2 of 5 examples
weave: Evaluated 3 of 5 examples
weave: Evaluated 4 of 5 examples
weave: Evaluated 5 of 5 examples
weave: Evaluation summary {
weave:   "exact_match": {
weave:     "exact_match": {
weave:       "true_count": 0,
weave:       "true_fraction": 0.0
weave:     }
weave:   },
weave:   "jaccard": {
weave:     "jaccard": {
weave:       "mean": 0.004347826086956522
weave:     }
weave:   },
weave:   "EmbeddingSimilarityScorer": {
weave:     "similarity_score": {
weave:       "mean": 0.6354363754361811
weave:     },
weave:     "is_similar": {
weave:       "true_count": 1,
weave:       "true_fraction": 0.2
weave:     }
weave:   },
weave:   "HallucinationFreeScorer": {
weave:     "has_hallucination": {
weave:       "true_count": 0,
weave:       "true_fraction": 0.0
weave:     }
weave:   },
weave:   "model_latency": {
weave:     "mean": 

**Result For Ground Model**

![1](./assets/1.png)

**Result For Loose Model**

![2](./assets/2.png)